# Week 59, Day 4: yield from and pipelines

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w059d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def inner():
    yield 1
    yield 2

def outer_loop():
    for x in inner():
        yield x
    yield 3

def outer_from():
    yield from inner()
    yield 3

def outer_wrong():
    yield inner()
    yield 3

print(list(outer_loop()), list(outer_from()))
print([type(x).__name__ for x in outer_wrong()])

In [ ]:
def chain_all(*iterables):
    for it in iterables:
        yield from it

def with_header(title, rows):
    yield f"== {title} =="
    yield from rows
    yield "== end =="

print(list(chain_all("ab", [1, 2], range(3))))
print("\n".join(with_header("stock", ["tea 4", "jam 0"])))

In [ ]:
def flatten(items):
    for item in items:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

def permutations(items):
    if len(items) <= 1:
        yield list(items)
        return
    for i, first in enumerate(items):
        for rest in permutations(items[:i] + items[i + 1:]):
            yield [first] + rest

print(list(flatten([1, [2, [3, [4]], 5], [], [[6]]])))
print(["".join(p) for p in permutations("abc")])

In [ ]:
def section(name, lines):
    count = 0
    for line in lines:
        yield f"{name}: {line}"
        count += 1
    return count

def document(parts):
    total = 0
    for name, lines in parts:
        total += yield from section(name, lines)
    yield f"({total} lines)"

for line in document([("intro", ["hello", "welcome"]), ("end", ["bye"])]):
    print(line)

In [ ]:
def strip_comments(lines):
    for line in lines:
        line = line.split("#")[0].strip()
        if line:
            yield line

def parse(lines):
    for line in lines:
        host, status, path = line.split()
        yield {"host": host, "status": int(status), "path": path}

def only_errors(records):
    return (r for r in records if r["status"] >= 400)

log = ["# web log", "10.0.0.1 200 /", "10.0.0.2 404 /missing  # oops", "10.0.0.1 500 /api", "", "10.0.0.3 200 /about"]
for record in only_errors(parse(strip_comments(log))):
    print(record)

> **Key idea.** yield from iterable yields every item of the iterable, then continues; it passes close() and friends through, and its value is the sub-generator's return value. It makes recursive generators natural: flatten nested lists, walk trees, list permutations, by yielding from the generator called on each part. A pipeline chains generator functions that each take an iterable and yield items, so every stage is small and testable and the whole runs in constant memory. Stages should do one job, never consume more than they need, keep their own state, and end in a sink that pulls values through.

### Exercise: Flattening anything

Write `flatten(items, depth=None)`: a recursive generator using `yield from` that yields the non-list items of a nested list in order; with `depth`, only that many levels are opened (deeper lists are yielded whole). Tuples and strings are **not** opened.

In [ ]:
def flatten(items, depth=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import types\ndef test_flatten():\n    \"yield from\"\n    assert list(flatten([1, [2, [3, [4]], 5], [], [[6]]])) == [1, 2, 3, 4, 5, 6]\n    assert list(flatten([1, [2, [3, [4]]]], depth=1)) == [1, 2, [3, [4]]] and list(flatten([[1]], depth=0)) == [[1]]\n    assert list(flatten([\"ab\", (\"c\", [1]), [[\"d\"]]])) == [\"ab\", (\"c\", [1]), \"d\"]\n    assert isinstance(flatten([]), types.GeneratorType) and \"yield from\" in source")

### Exercise: Walking a family tree

A family tree is a dict `{"name": ..., "children": [...]}` with the same shape for each child. Write recursive generators with `yield from`: `names(person)`: everyone in pre-order (a person, then each child's family in order); `with_depth(person)`: `(name, depth)` pairs, the root at 0; and `lines(person)`: the tree drawn as lines, each name indented by two spaces per level.

In [ ]:
def names(person):
    ___

def with_depth(person, depth=0):
    ___

def lines(person):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tree():\n    \"recursive generators\"\n    fam = {\"name\": \"Ada\", \"children\": [{\"name\": \"Ben\", \"children\": [{\"name\": \"Cy\", \"children\": []}]}, {\"name\": \"Di\", \"children\": []}]}\n    assert list(names(fam)) == [\"Ada\", \"Ben\", \"Cy\", \"Di\"]\n    assert list(with_depth(fam)) == [(\"Ada\", 0), (\"Ben\", 1), (\"Cy\", 2), (\"Di\", 1)]\n    assert list(lines(fam)) == [\"Ada\", \"  Ben\", \"    Cy\", \"  Di\"] and list(names({\"name\": \"Solo\", \"children\": []})) == [\"Solo\"]\n    assert source.count(\"yield from\") >= 2")

### Exercise: Every arrangement, lazily

Write recursive generators: `permutations(items)` (lists, in the order of choosing each item first in turn), `subsets(items)` (lists: all the subsets of the items after the first, then all of those again with the first item added at the front), and `compositions(n)`: every way to write n as an ordered sum of positive whole numbers, as lists (`3`: `[1, 1, 1]`, `[1, 2]`, `[2, 1]`, `[3]`, first part smallest first).

In [ ]:
def permutations(items):
    ___

def subsets(items):
    ___

def compositions(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_combos():\n    \"recursion and yield from\"\n    assert [list(p) for p in itertools.permutations(\"abc\")] == [list(p) for p in permutations(list(\"abc\"))]\n    assert list(subsets([1, 2])) == [[], [2], [1], [1, 2]] and list(subsets([])) == [[]]\n    assert sorted(map(sorted, subsets([1, 2, 3]))) == sorted(sorted(c) for r in range(4) for c in itertools.combinations([1, 2, 3], r))\n    assert list(compositions(3)) == [[1, 1, 1], [1, 2], [2, 1], [3]] and len(list(compositions(10))) == 512\n    assert next(permutations(list(range(50)))) == list(range(50))")

### Exercise: Reporting back

Write `numbered(lines, start)`: a generator yielding `"{n}. {line}"` for each line, numbering from `start`, and **returning** the next number to use. Then `chapters(book)`: for a list of `(title, lines)`, yield each title in capitals, then its numbered lines, with the numbering **continuing** across chapters (collect each `numbered` call's return value with `yield from`).

In [ ]:
def numbered(lines, start):
    ___

def chapters(book):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_returns():\n    \"yield from's value\"\n    g = numbered([\"a\", \"b\"], 5)\n    assert next(g) == \"5. a\" and next(g) == \"6. b\"\n    try:\n        next(g)\n        assert False\n    except StopIteration as stop:\n        assert stop.value == 7\n    book = [(\"one\", [\"x\", \"y\"]), (\"two\", []), (\"three\", [\"z\"])]\n    assert list(chapters(book)) == [\"ONE\", \"1. x\", \"2. y\", \"TWO\", \"THREE\", \"3. z\"]\n    assert \"yield from\" in source")

### Exercise: Stages of a pipeline

Write three stage generators, each taking an iterable and yielding: `dedupe(items, key=None)` (each item whose key hasn't been seen before; `key` defaults to the item itself), `changes(items)` (each item that differs from the one before it, the first always included), and `numbered(items, start=1)` (`(n, item)` pairs). Then `pipeline(items)`: `numbered(dedupe(changes(items), key=str.lower))`, returned as a generator. All lazy.

In [ ]:
def dedupe(items, key=None):
    ___

def changes(items):
    ___

def numbered(items, start=1):
    ___

def pipeline(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, types\ndef test_stages():\n    \"small stages\"\n    assert list(dedupe([3, 1, 3, 2, 1])) == [3, 1, 2] and list(dedupe([\"A\", \"a\", \"b\"], key=str.lower)) == [\"A\", \"b\"]\n    assert list(changes([1, 1, 2, 2, 2, 1, 3, 3])) == [1, 2, 1, 3] and list(changes([])) == []\n    assert list(numbered(\"ab\", 5)) == [(5, \"a\"), (6, \"b\")]\n    words = [\"Tea\", \"tea\", \"TEA\", \"milk\", \"Milk\", \"tea\", \"jam\"]\n    assert list(pipeline(words)) == [(1, \"Tea\"), (2, \"milk\"), (3, \"jam\")] and isinstance(pipeline([]), types.GeneratorType)\n    assert list(itertools.islice(dedupe(itertools.cycle([1, 2, 3, 4])), 4)) == [1, 2, 3, 4]\n    assert list(itertools.islice(changes(itertools.count()), 3)) == [0, 1, 2]")

### Bug hunt: Yielding the generator

`yield (...)` hands out each generator object instead of its values, so this prints `['generator', 'generator']`. Use `yield from` so it prints `[2, 4, 1, 3]`.

In [ ]:
def evens_then_odds(nums):
    yield (n for n in nums if n % 2 == 0)
    yield (n for n in nums if n % 2)

print([x if isinstance(x, int) else type(x).__name__ for x in evens_then_odds([1, 2, 3, 4])])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"yield from\"\n    expect_output(\"[2, 4, 1, 3]\")")

### Bug hunt: The recursive call that does nothing

Calling `flatten(item)` on a nested list makes a generator and throws it away, so the nested items vanish and this prints `[1, 5]`. Use `yield from` on the recursive call so it prints `[1, 2, 3, 4, 5]`.

In [ ]:
def flatten(items):
    for item in items:
        if isinstance(item, list):
            flatten(item)
        else:
            yield item

print(list(flatten([1, [2, [3, 4]], 5])))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"delegate\"\n    expect_output(\"[1, 2, 3, 4, 5]\")")

### Bug hunt: A stage that eats everything

This stage calls `list(source)`, which reads the **whole** endless stream before yielding anything, so the pipeline never produces its first value. (It's cut off here by a limit, and prints `stuck`.) Loop over the source directly so it prints `[0, 2, 4]`.

In [ ]:
import itertools

def doubled(source):
    items = list(itertools.islice(source, 10 ** 5))
    if len(items) == 10 ** 5:
        yield "stuck"
        return
    for x in items:
        yield x * 2

result = list(itertools.islice(doubled(itertools.count()), 3))
print("stuck" if "stuck" in result else result)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stay lazy\"\n    expect_output(\"[0, 2, 4]\")")

### Puzzle: A recursive flatten

Arrange the lines of a recursive generator that flattens nested lists. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # def g(): yield from "hi"; yield "!"   -- then list(g())
q = ___   # how many permutations permutations("abcd") yields
r = ___   # def s(): yield 1; return "done"   -- the value of StopIteration after one next() too many
s = ___   # list(changes("aabbba"))
t = ___   # True or False: yield from works only with other generators

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"h\", \"i\", \"!\"] and q == 24 and r == \"done\", \"p, q and r\"\n    assert s == [\"a\", \"b\", \"a\"] and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a log analyser in stages

A web server log has lines `host status path bytes`, for example `10.0.0.1 200 /index.html 512`, with `#` comments and occasional broken lines. Build a pipeline of generator stages and some sinks:

- Stages: `parse(lines)` yields dicts with keys `host`, `status` (int), `path`, `bytes` (int), skipping comments, blank and broken lines; `where(records, **conditions)` yields records matching every given field exactly (`where(r, status=404)`); `bursts(records, size)` yields lists of `size` consecutive **error** records (status 500 or more) that occur together, with no non-error record between them (a run of 5 errors with size 2 gives two bursts: the first two and the next two).
- Sinks: `status_counts(lines)` (a dict from status to count, via the stages), `top_paths(lines, n)` (the n paths with most bytes served in total, most first, ties alphabetical, as `(path, bytes)`), and `first_burst(lines, size)` (the first burst, or `None`, reading no further than needed; the checks give it an endless log).

In [ ]:
from collections import Counter

def parse(lines):
    ___

def where(records, **conditions):
    ___

def bursts(records, size):
    ___

def status_counts(lines):
    ___

def top_paths(lines, n):
    ___

def first_burst(lines, size):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, types\nLOG = [\"# log\", \"10.0.0.1 200 /index.html 512\", \"10.0.0.2 404 /missing 0\", \"bad line\", \"10.0.0.1 500 /api 20\",\n       \"10.0.0.3 503 /api 10\", \"\", \"10.0.0.1 200 /index.html 512\", \"10.0.0.4 500 /api 5\", \"10.0.0.4 200 /about x\"]\ndef test_stages():\n    \"stages\"\n    recs = list(parse(LOG))\n    assert len(recs) == 6 and recs[0] == {\"host\": \"10.0.0.1\", \"status\": 200, \"path\": \"/index.html\", \"bytes\": 512}\n    assert isinstance(parse(LOG), types.GeneratorType)\n    assert [r[\"path\"] for r in where(parse(LOG), status=200)] == [\"/index.html\", \"/index.html\"]\n    assert [r[\"status\"] for r in where(parse(LOG), host=\"10.0.0.1\", status=500)] == [500]\n    assert [[r[\"status\"] for r in b] for b in bursts(parse(LOG), 2)] == [[500, 503]]\n    assert len(list(bursts(parse(LOG), 1))) == 3\ndef test_sinks():\n    \"sinks\"\n    assert status_counts(iter(LOG)) == {200: 2, 404: 1, 500: 2, 503: 1}\n    assert top_paths(LOG, 2) == [(\"/index.html\", 1024), (\"/api\", 35)] and top_paths([], 3) == []\n    def endless():\n        yield \"1.1.1.1 200 / 1\"\n        yield \"1.1.1.1 500 /x 1\"\n        yield \"1.1.1.1 200 / 1\"\n        yield \"2.2.2.2 502 /y 1\"\n        yield \"3.3.3.3 504 /z 1\"\n        while True:\n            yield \"4.4.4.4 200 / 1\"\n    b = first_burst(endless(), 2)\n    assert [r[\"host\"] for r in b] == [\"2.2.2.2\", \"3.3.3.3\"] and first_burst(LOG, 3) is None")

## Warm-up

`yield from inner()` does what the loop does: passes on every value. Plain `yield inner()` hands out the generator object itself, a common slip.

## Handing over

`yield from iterable` yields every item of the iterable, in order, then carries on with the next line. It works with any iterable (a list, a string, a range, another generator), so `yield from "abc"` yields three characters.

For plain iteration it's shorthand for `for x in iterable: yield x`, but it does more under the hood: if the consumer calls `close()` or uses the advanced `send()` and `throw()` methods, they pass straight through to the inner generator, and the inner generator's **return value** becomes the value of the `yield from` expression (below). PEP 380 (Gregory Ewing, 2009) added it in Python 3.3.

## Recursive generators

`yield from` makes recursion and generators fit together. A generator that walks a structure calls **itself** on the parts and passes on what they yield:

- **Flattening** a nested list: for each item, if it's a list, `yield from flatten(item)`; otherwise `yield item`.
- **Walking a tree**: yield the node, then `yield from` each child's walk.
- **Combinatorial sequences**: permutations of a list are each first item followed by each permutation of the rest.

Each item travels up through one `yield from` per level, so very deep structures are slower than an explicit stack (and still hit the recursion limit near 1000 levels), but for ordinary nesting this is the clearest code there is.

## Return values

A generator may `return value` at the end. A normal loop ignores it, but `result = yield from sub()` passes on everything `sub()` yields **and** then receives its return value. That lets a sub-generator report a summary (how many items it produced, a total, a final state) to the generator that delegated to it.

## Pipelines of stages

A **pipeline** is a chain of generator functions, each taking an iterable and yielding a new stream:

```
lines → strip_comments → parse → only_errors → count_by_host
```

Each stage is a few lines and does one thing, so it can be tested alone with a short list. Connected, they process an enormous file or an endless stream in constant memory, because each record flows through all the stages before the next is read. Wednesday's generator expressions are one-line stages; generator functions take over when a stage needs `try`, state, or several statements. This is the Unix pipe idea (`grep | sort | uniq`, Douglas McIlroy, 1973) inside a Python program.

## Designing stages

Good stages have a few properties in common:

- **Take an iterable, yield items.** Then any stage can follow any other with matching items, and lists work in tests.
- **Do one job.** "Parse and filter and count" is three stages.
- **Don't consume more than you need.** A stage that calls `list(source)` breaks the laziness of the whole pipeline.
- **Put state in the stage that needs it**: a stage that removes duplicates keeps a set of what it has seen; a stage that numbers items keeps a counter.
- **End with a sink**: something that pulls values through: `for`, `sum`, a `Counter`, `next`, writing to a file. Until then, nothing runs.

Stages that need the **previous** item (differences, change detection) keep it in a local variable; stages that group neighbouring items keep the current group and yield it when it ends (tomorrow's `itertools.groupby` does exactly that).

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a pipeline you could use on data you care about (bank statements, a playlist, a reading log): the source, each stage, and the sink. Tomorrow: `itertools`, the standard library's box of ready-made stages.